# OKX Spot Historical Data Import

This notebook imports historical OHLCV data from OKX Spot markets into the PostgreSQL database.

**Data Source:** OKX Public API (https://www.okx.com/docs-v5/en/#rest-api-market-data)

**Features:**
- Fetches historical klines/candlestick data via REST API
- Automatic pagination for large date ranges
- Retry logic with exponential backoff
- Rate limiting to respect API constraints
- Duplicate detection
- Bulk insert operations

**Data Format:**
- API response: [timestamp_ms, open, high, low, close, volume, volume_currency, volume_quote, confirm]
- Timestamps are in milliseconds (UNIX epoch)
- Max 100 candles per request

**Rate Limits:**
- Public API: 20 requests/2 seconds
- We use conservative 0.2s delay between requests (5 req/sec)

In [ ]:
import pandas as pd
import time

from clients.db_client import DBClient
from clients.okx_client import OKXClient

In [ ]:
# Configuration
INTERVAL = "1H"  # Options: 1m, 3m, 5m, 15m, 30m, 1H, 2H, 4H, 6H, 12H, 1D, 1W, 1M
START_DATE = "2023-01-01"
END_DATE = "2025-12-31"

okx = OKXClient(request_delay=0.2, max_retries=3)

In [ ]:
# Test download
test_df = okx.download_klines("BTC-USDT", INTERVAL, "2023-01-01", "2025-12-31")
print(f"\nFirst few rows:")
print(test_df.head())
print(f"\nLast few rows:")
print(test_df.tail())
print(f"\nDate range: {test_df['timestamp'].min()} to {test_df['timestamp'].max()}")
print(f"Total rows: {len(test_df)}")

In [ ]:
with DBClient() as db:
        
    # Import OKX data
    total_inserted = 0
    total_skipped = 0

    # Get all OKX instruments from database
    instruments_dict = db.get_instruments_by_exchange(exchange="okx")
    print(f"Found {len(instruments_dict)} OKX instruments in database\n")

    if not instruments_dict:
        print("No OKX instruments found! Please run sql/seed_okx_spot.sql first.")

    for ticker, instrument_id in instruments_dict.items():    
        print(f"Processing {ticker} (instrument_id: {instrument_id})")
        
        df = okx.download_klines(ticker, INTERVAL, START_DATE, END_DATE)
        
        if df.empty:
            print("No data downloaded, skipping")
            continue
        
        # Check existing data in database
        db_min, db_max = db.get_timestamp_range(instrument_id)
        
        # Check for duplicates
        if db_min and db_max:
            # Filter out overlapping data
            df = df[(df['timestamp'] < db_min) | (df['timestamp'] > db_max)]
            
            if df.empty:
                print(f"All data already exists, skipping")
                total_skipped += 1
                continue
        
        # Insert data
        df['instrument_id'] = instrument_id
        rows = db.insert_market_data(df)
        total_inserted += rows
        print(f"Inserted {rows} rows")
        
        # Small delay between instruments
        time.sleep(0.5)

    print(f"Total rows inserted: {total_inserted:,}")
    print(f"Total instruments skipped: {total_skipped}")